### Overview

This notebook implements a lightweight rule-based language filtering pipeline
for conversational tree-structured data (e.g., ShareGPT-style datasets).

Goal:
- Load raw conversation trees
- Detect English prompts with deterministic heuristics
- Filter non-English trees
- Save the cleaned dataset
- Inspect random sample prompts

This version separates:
- configuration
- processing logic
- execution

In [11]:
import re
import json
import random
from pathlib import Path
from time import perf_counter

def find_project_root(start=None, marker="01_data"):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root containing '{marker}'")

PROJECT_ROOT = find_project_root()

CONFIG = {
    "data_dir_raw": PROJECT_ROOT / "01_data" / "01_raw" / "messy_dataset",
    "data_dir_processed": PROJECT_ROOT / "01_data" / "02_processed",
    "input_files": [
        "sg_90k_part1.json",
        "sg_90k_part2.json",
    ],
    "output_file": "sharegpt_english.json",
    "min_words": 5,
    "min_english_stopwords": 2,
    "min_margin": 1,
    "sample_size": 10,
    "sample_max_chars": 700,
    "random_seed": 42,
}

In [12]:
WORD_RE = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ']+")

STOPWORDS = {
    "en": {
        "the","and","to","of","a","in","that","is","it","for","on","with","as","this",
        "be","are","you","your","can","will","not","or","if","we","from","by","at","an",
        "have","has","do","does","what","which","when","where","how","why","about","would",
        "could","should","there","their","they","them","these","those","one","more","use",
        "using","make","write","explain","please","help"
    },
    "de": {
        "der","die","das","und","ist","ich","du","sie","er","es","wir","nicht","ein","eine",
        "zu","mit","auf","für","von","wie","was","warum","wenn","dass","kann","sind","haben",
        "werden","auch","oder","aber","bitte","über","hallo"
    },
    "nl": {
        "de","het","een","en","van","ik","je","niet","dat","dit","die","op","voor","met",
        "als","zijn","hebben","worden","wat","waar","waarom","hoe"
    },
    "vi": {
        "và","là","của","có","cho","trong","một","không","tôi","bạn","hãy","này","để"
    },
    "es": {
        "el","la","los","las","y","de","que","en","un","una","es","por","para","con","como","no","se"
    },
    "fr": {
        "le","la","les","et","de","des","du","un","une","est","dans","pour","avec","que","qui"
    },
    "pt": {
        "o","a","os","as","e","de","que","em","um","uma","é","para","com","como","não"
    },
    "it": {
        "il","lo","la","gli","le","e","di","che","in","un","una","è","per","con"
    },
}

In [13]:
def get_first_user_prompt(tree):
    conversations = tree.get("conversations", [])

    for message in conversations:
        if message.get("from") == "human":
            return message.get("value", "")

    return ""


def normalize_words(text):
    return [w.lower().strip("'") for w in WORD_RE.findall(text)]


def count_stopwords(words, stopwords_dict):
    return {
        lang: sum(w in stopwords for w in words)
        for lang, stopwords in stopwords_dict.items()
    }


def get_language_scores_from_prompt(prompt, stopwords_dict):
    words = normalize_words(prompt)
    stopword_scores = count_stopwords(words, stopwords_dict)

    english_score = stopword_scores["en"]
    strongest_other = max(v for k, v in stopword_scores.items() if k != "en")

    return {
        "words": words,
        "word_count": len(words),
        "stopword_scores": stopword_scores,
        "english_score": english_score,
        "strongest_other": strongest_other,
    }


def is_english_tree(tree, config, stopwords_dict):
    prompt = get_first_user_prompt(tree)
    scores = get_language_scores_from_prompt(prompt, stopwords_dict)

    if scores["word_count"] < config["min_words"]:
        return False

    english_score = scores["english_score"]
    strongest_other = scores["strongest_other"]

    return (
        english_score >= config["min_english_stopwords"]
        and (english_score - strongest_other) >= config["min_margin"]
    )


def filter_english_trees(data, config, stopwords_dict):
    return [
        tree for tree in data
        if is_english_tree(tree, config, stopwords_dict)
    ]

In [14]:
def load_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


def load_all_json(paths):
    data = []
    for path in paths:
        data.extend(load_json(path))
    return data


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)

In [15]:
def show_random_questions(data, n=10, max_chars=700, seed=None):
    if seed is not None:
        random.seed(seed)

    sample_size = min(n, len(data))

    for i in random.sample(range(len(data)), sample_size):
        tree = data[i]
        prompt = get_first_user_prompt(tree)

        print("=" * 100)
        print(f"Index: {i}")
        print(prompt[:max_chars])

In [16]:
input_paths = [
    CONFIG["data_dir_raw"] / filename
    for filename in CONFIG["input_files"]
]

output_path = CONFIG["data_dir_processed"] / CONFIG["output_file"]

print("Input paths:")
for path in input_paths:
    print("-", path)

print("\nOutput path:")
print("-", output_path)

Input paths:
- c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\01_raw\messy_dataset\sg_90k_part1.json
- c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\01_raw\messy_dataset\sg_90k_part2.json

Output path:
- c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\02_processed\sharegpt_english.json


In [17]:
start = perf_counter()

data = load_all_json(input_paths)
english_data = filter_english_trees(data, CONFIG, STOPWORDS)

duration = perf_counter() - start

save_json(english_data, output_path)

print(f"Original: {len(data)}")
print(f"English only: {len(english_data)}")
print(f"Removed: {len(data) - len(english_data)}")
print(f"Time: {duration:.2f}s")
print(f"Trees/sec: {len(data)/duration:.0f}")

Original: 90665
English only: 58921
Removed: 31744
Time: 21.47s
Trees/sec: 4223


In [18]:
show_random_questions(
    english_data,
    n=CONFIG["sample_size"],
    max_chars=CONFIG["sample_max_chars"],
    seed=CONFIG["random_seed"],
)

Index: 41905
Hong Kong STEM Education Centre hosts STEM fair On March 1st, 2023, the HK STEM Education Centre, an education organization that has earned the trust of many customers through its excellent services, announced the grand ceremony opening of its fair. The grand ceremony opening will be held on April 1st, 2023, from 10:00 am to 11:30 am at Hall 2, Central Exhibition Centre. The fair will offer a wide variety of events including scientific demonstrations, workshops, and competitions .from April 1st to April 2nd, 2023, at the same place where the ceremony opened. Expressing his enthusiasm, Dr. Fred Liu, Chairperson of Hong Kong STEM Education Centre, commented, "I am excited about the event becau
Index: 7296
Please tell us about the spam sniper solution of Jiranjigyo Soft in Korea.

Please write in English language.
    
        
            
                지금 번역하기
            
        
    

Index: 1639
Where’s the gold at?
Index: 48598
What are some common constructions in t